In [33]:
# IndiaPII - Dataset Generation

import os
import json
import random
import pandas as pd
import numpy as np

from pathlib import Path

# Reproducibility
random.seed(42)
np.random.seed(42)

# Project directories
PROJECT_ROOT = Path.cwd().parent

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Supported entity labels
LABELS = [
    "PERSON",
    "ADDRESS",
    "PHONE",
    "AADHAAR",
    "PAN",
    "EMAIL",
    "UPI"
]

print("IndiaPII Dataset Environment Ready!")
print("Labels:", LABELS)
print("Raw directory:", RAW_DIR)
print("Processed directory:", PROCESSED_DIR)



IndiaPII Dataset Environment Ready!
Labels: ['PERSON', 'ADDRESS', 'PHONE', 'AADHAAR', 'PAN', 'EMAIL', 'UPI']
Raw directory: /Users/navyaa09/IndiaPII/data/raw
Processed directory: /Users/navyaa09/IndiaPII/data/processed


# IndiaPII – Synthetic Dataset Generation

This notebook generates a multilingual synthetic dataset
for India-specific Personally Identifiable Information (PII)
detection.

Supported languages:
- English
- Hindi
- Hinglish

Supported entity types:
PERSON, ADDRESS, PHONE, AADHAAR, PAN, EMAIL, UPI

All generated personal information is synthetic.

In [34]:

import random
import json
import re
import pandas as pd
from pathlib import Path

random.seed(42)

# Synthetic entity pools
DATA = {
    "PERSON": [
        "Rahul Sharma", "Priya Mehta", "Ananya Rao",
        "Aman Verma", "Neha Gupta", "Karan Malhotra",
        "Riya Singh", "Arjun Kumar", "Kavya Nair",
        "Amit Patel", "रोहित शर्मा", "प्रिया मेहता",
        "अमित कुमार", "नेहा गुप्ता", "राहुल वर्मा",
        "रोहित", "पूजा सिंह", "सीमा देवी"
    ],
    "ADDRESS": [
        "24 MG Road, Bengaluru",
        "Sector 15, Noida",
        "Flat 302, Whitefield, Bengaluru",
        "12 Park Street, Kolkata",
        "HSR Layout, Bengaluru",
        "Indiranagar, Bengaluru",
        "एमजी रोड, भोपाल",
        "सेक्टर 15, नोएडा",
        "व्हाइटफील्ड, बेंगलुरु",
        "इंदिरानगर, बेंगलुरु"
    ],
    "PHONE": [
        "9000000001", "9000000002", "9000000003",
        "9000000004", "9000000005", "9000000006",
        "9000000007", "9000000008", "9000000009",
        "9000000010"
    ],
    "AADHAAR": [
        "0000 0000 0001", "0000 0000 0002",
        "0000 0000 0003", "0000 0000 0004",
        "0000 0000 0005", "0000 0000 0006"
    ],
    "PAN": [
        "ABCDE1234F", "PQRSX5678K",
        "FGHIJ4567L", "LMNOP1234Q",
        "QRSTU5678A", "VWXYZ1234B"
    ],
    "EMAIL": [
        "rahul.test@example.com",
        "priya.test@example.com",
        "ananya@example.org",
        "aman.work@example.net",
        "neha.demo@example.com",
        "riya.test@example.org"
    ],
    "UPI": [
        "rahul@okaxis", "priya@ybl",
        "ananya@oksbi", "aman@paytm",
        "neha@okhdfcbank", "riya@ibl"
    ]
}

# Templates: language, sentence, entity labels
TEMPLATES = [
    # ---------------- ENGLISH ----------------
    ("en", "My name is {PERSON} and my phone number is {PHONE}.", ["PERSON", "PHONE"]),
    ("en", "Please send the documents to {EMAIL}.", ["EMAIL"]),
    ("en", "The applicant {PERSON} lives at {ADDRESS}.", ["PERSON", "ADDRESS"]),
    ("en", "My Aadhaar number is {AADHAAR}.", ["AADHAAR"]),
    ("en", "The PAN registered to {PERSON} is {PAN}.", ["PERSON", "PAN"]),
    ("en", "Transfer the payment to {UPI}.", ["UPI"]),
    ("en", "You can contact {PERSON} at {PHONE} or {EMAIL}.", ["PERSON", "PHONE", "EMAIL"]),
    ("en", "My current address is {ADDRESS}.", ["ADDRESS"]),
    ("en", "The UPI ID of {PERSON} is {UPI}.", ["PERSON", "UPI"]),
    ("en", "My registered phone is {PHONE} and PAN is {PAN}.", ["PHONE", "PAN"]),

    # ---------------- HINDI ----------------
    ("hi", "मेरा नाम {PERSON} है और मेरा मोबाइल नंबर {PHONE} है।", ["PERSON", "PHONE"]),
    ("hi", "कृपया इस ईमेल पर संपर्क करें {EMAIL}।", ["EMAIL"]),
    ("hi", "{PERSON} का पता {ADDRESS} है।", ["PERSON", "ADDRESS"]),
    ("hi", "मेरा आधार नंबर {AADHAAR} है।", ["AADHAAR"]),
    ("hi", "{PERSON} का पैन नंबर {PAN} है।", ["PERSON", "PAN"]),
    ("hi", "पैसे इस UPI ID पर भेजें {UPI}।", ["UPI"]),
    ("hi", "{PERSON} से {PHONE} पर संपर्क करें।", ["PERSON", "PHONE"]),
    ("hi", "मेरा वर्तमान पता {ADDRESS} है।", ["ADDRESS"]),
    ("hi", "{PERSON} की UPI ID {UPI} है।", ["PERSON", "UPI"]),
    ("hi", "मेरा फोन नंबर {PHONE} और ईमेल {EMAIL} है।", ["PHONE", "EMAIL"]),

    # ---------------- HINGLISH ----------------
    ("hinglish", "Mera naam {PERSON} hai aur mera number {PHONE} hai.", ["PERSON", "PHONE"]),
    ("hinglish", "Please documents {EMAIL} par send kar dena.", ["EMAIL"]),
    ("hinglish", "Main {ADDRESS} mein rehta hoon.", ["ADDRESS"]),
    ("hinglish", "Mera Aadhaar number {AADHAAR} hai.", ["AADHAAR"]),
    ("hinglish", "{PERSON} ka PAN number {PAN} hai.", ["PERSON", "PAN"]),
    ("hinglish", "Payment {UPI} par kar dena.", ["UPI"]),
    ("hinglish", "{PERSON} ko {PHONE} par call karna.", ["PERSON", "PHONE"]),
    ("hinglish", "Mera current address {ADDRESS} hai.", ["ADDRESS"]),
    ("hinglish", "{PERSON} ki UPI ID {UPI} hai.", ["PERSON", "UPI"]),
    ("hinglish", "Meri email {EMAIL} aur phone {PHONE} hai.", ["EMAIL", "PHONE"]),

    # -------- ADDITIONAL ENGLISH AADHAAR TEMPLATES --------
    ("en", "Please verify my Aadhaar: {AADHAAR}.", ["AADHAAR"]),
    ("en", "I have provided my Aadhaar number {AADHAAR}.", ["AADHAAR"]),
    ("en", "The Aadhaar ID linked to my account is {AADHAAR}.", ["AADHAAR"]),
    ("en", "My identification number is {AADHAAR}.", ["AADHAAR"]),
    ("en", "Aadhaar details: {AADHAAR}.", ["AADHAAR"]),

    # -------- ADDITIONAL HINDI AADHAAR TEMPLATES --------
    ("hi", "कृपया मेरा आधार नंबर {AADHAAR} सत्यापित करें।", ["AADHAAR"]),
    ("hi", "मेरा आधार संख्या {AADHAAR} है।", ["AADHAAR"]),
    ("hi", "पहचान के लिए आधार नंबर {AADHAAR} दर्ज करें।", ["AADHAAR"]),
    ("hi", "आधार विवरण: {AADHAAR}।", ["AADHAAR"]),
    ("hi", "यह मेरा आधार कार्ड नंबर {AADHAAR} है।", ["AADHAAR"]),

    # -------- ADDITIONAL HINGLISH AADHAAR TEMPLATES --------
    ("hinglish", "Please mera Aadhaar verify karo: {AADHAAR}.", ["AADHAAR"]),
    ("hinglish", "Meri Aadhaar ID {AADHAAR} hai.", ["AADHAAR"]),
    ("hinglish", "Account ke saath linked Aadhaar {AADHAAR} hai.", ["AADHAAR"]),
    ("hinglish", "Aadhaar details check karo {AADHAAR}.", ["AADHAAR"]),
    ("hinglish", "Mera identification number {AADHAAR} hai.", ["AADHAAR"])
]


def generate_example(template, language, labels, template_id):
    """Fill a template and calculate exact PII character offsets."""

    values = {
        label: random.choice(DATA[label])
        for label in labels
    }

    text = template.format(**values)
    entities = []

    for label, value in values.items():
        start = text.index(value)
        end = start + len(value)

        entities.append({
            "start": start,
            "end": end,
            "label": label
        })

    entities.sort(key=lambda x: x["start"])

    return {
        "text": text,
        "entities": entities,
        "language": language,
        "template_id": template_id
    }


# Generate 900 synthetic examples: 20 per template
dataset = []

SAMPLES_PER_TEMPLATE = 20

for i in range(len(TEMPLATES) * SAMPLES_PER_TEMPLATE):
    template_id = i % len(TEMPLATES)

    language, template, labels = TEMPLATES[template_id]

    example = generate_example(
        template,
        language,
        labels,
        template_id
    )

    dataset.append(example)

random.shuffle(dataset)

# Verify entity offsets
for item in dataset:
    for entity in item["entities"]:
        extracted = item["text"][entity["start"]:entity["end"]]
        assert extracted in DATA[entity["label"]], (
            f"Invalid entity offset: {item['text']} | {entity}"
        )

print("Dataset generated successfully!")
print("Total templates:", len(TEMPLATES))
print("Total sentences:", len(dataset))

print("\nLanguages:")
print(pd.Series(
    [x["language"] for x in dataset]
).value_counts().to_dict())

print("\nEntity distribution:")

entity_counts = {}

for item in dataset:
    for entity in item["entities"]:
        entity_counts[entity["label"]] = (
            entity_counts.get(entity["label"], 0) + 1
        )

print(entity_counts)

print("\nAadhaar template count:")
aadhaar_template_ids = {
    item["template_id"]
    for item in dataset
    if any(e["label"] == "AADHAAR" for e in item["entities"])
}
print(len(aadhaar_template_ids))

print("\nSample:")
print(json.dumps(dataset[0], ensure_ascii=False, indent=2))


Dataset generated successfully!
Total templates: 45
Total sentences: 900

Languages:
{'hinglish': 300, 'en': 300, 'hi': 300}

Entity distribution:
{'PERSON': 280, 'PAN': 80, 'AADHAAR': 360, 'EMAIL': 120, 'UPI': 120, 'ADDRESS': 120, 'PHONE': 180}

Aadhaar template count:
18

Sample:
{
  "text": "Kavya Nair ka PAN number LMNOP1234Q hai.",
  "entities": [
    {
      "start": 0,
      "end": 10,
      "label": "PERSON"
    },
    {
      "start": 25,
      "end": 35,
      "label": "PAN"
    }
  ],
  "language": "hinglish",
  "template_id": 24
}


In [35]:

# Step 3: Save the synthetic dataset

import json
from pathlib import Path

# Save location
output_file = RAW_DIR / "synthetic_pii_dataset.json"

# Save complete dataset
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(
        dataset,
        f,
        ensure_ascii=False,
        indent=2
    )

print("Dataset saved successfully!")
print("File:", output_file)
print("Total records:", len(dataset))


Dataset saved successfully!
File: /Users/navyaa09/IndiaPII/data/raw/synthetic_pii_dataset.json
Total records: 900


In [36]:

# Validate entity spans and dataset structure

validation_errors = []

for idx, item in enumerate(dataset):

    text = item["text"]

    # Validate record structure
    if not all(key in item for key in ["text", "entities", "language", "template_id"]):
        validation_errors.append(f"Missing fields in record {idx}")
        continue

    for entity in item["entities"]:

        start = entity["start"]
        end = entity["end"]
        label = entity["label"]

        # Validate span boundaries
        if start < 0 or end > len(text) or start >= end:
            validation_errors.append(
                f"Invalid span in record {idx}: {start}-{end}"
            )
            continue

        extracted_text = text[start:end]

        # Validate entity label
        if label not in LABELS:
            validation_errors.append(
                f"Unknown label in record {idx}: {label}"
            )
            continue

        # Validate non-empty entity
        if not extracted_text.strip():
            validation_errors.append(
                f"Empty entity in record {idx}"
            )

        # Validate extracted entity against its pool
        if extracted_text not in DATA[label]:
            validation_errors.append(
                f"Entity mismatch in record {idx}: {extracted_text}"
            )

print("Total records:", len(dataset))
print("Validation errors:", len(validation_errors))

if validation_errors:
    print("\nFirst 10 errors:")
    print(validation_errors[:10])
else:
    print("All entity annotations are valid!")

assert len(dataset) == 900, "Expected 900 generated samples"
assert not validation_errors, "Dataset validation failed"


Total records: 900
Validation errors: 0
All entity annotations are valid!


In [37]:

# Preview dataset

df = pd.DataFrame(dataset)

display(df[["text", "language", "entities"]].head(10))

print("\nLanguage distribution:")
display(df["language"].value_counts())

print("\nEntity distribution:")

entity_distribution = {}

for item in dataset:
    for entity in item["entities"]:
        label = entity["label"]
        entity_distribution[label] = (
            entity_distribution.get(label, 0) + 1
        )

display(pd.DataFrame(
    list(entity_distribution.items()),
    columns=["Entity", "Count"]
))


,text,language,entities
0,Kavya Nair ka PAN number LMNOP1234Q hai.,hinglish,"[{'start': 0, 'end': 10, 'label': 'PERSON'}, {..."
1,My identification number is 0000 0000 0003.,en,"[{'start': 28, 'end': 42, 'label': 'AADHAAR'}]"
2,Please send the documents to neha.demo@example...,en,"[{'start': 29, 'end': 50, 'label': 'EMAIL'}]"
3,मेरा आधार नंबर 0000 0000 0006 है।,hi,"[{'start': 15, 'end': 29, 'label': 'AADHAAR'}]"
4,Amit Patel ki UPI ID priya@ybl hai.,hinglish,"[{'start': 0, 'end': 10, 'label': 'PERSON'}, {..."
5,"The applicant रोहित lives at व्हाइटफील्ड, बेंग...",en,"[{'start': 14, 'end': 19, 'label': 'PERSON'}, ..."
6,"The applicant अमित कुमार lives at Indiranagar,...",en,"[{'start': 14, 'end': 24, 'label': 'PERSON'}, ..."
7,कृपया मेरा आधार नंबर 0000 0000 0002 सत्यापित क...,hi,"[{'start': 21, 'end': 35, 'label': 'AADHAAR'}]"
8,My registered phone is 9000000005 and PAN is V...,en,"[{'start': 23, 'end': 33, 'label': 'PHONE'}, {..."
9,Aadhaar details check karo 0000 0000 0001.,hinglish,"[{'start': 27, 'end': 41, 'label': 'AADHAAR'}]"



Language distribution:


language
hinglish    300
en          300
hi          300
Name: count, dtype: int64


Entity distribution:


,Entity,Count
0,PERSON,280
1,PAN,80
2,AADHAAR,360
3,EMAIL,120
4,UPI,120
5,ADDRESS,120
6,PHONE,180


In [38]:

# Step 4.1: Character-level annotations to BIO tags

import unicodedata

BIO_LABELS = ["O"] + [
    prefix + "-" + label
    for label in LABELS
    for prefix in ["B", "I"]
]


def tokenize_with_offsets(text):
    """
    Unicode-aware tokenization.
    Preserves Hindi combining marks and character offsets.
    """

    tokens = []
    current_start = None

    for i, char in enumerate(text):

        category = unicodedata.category(char)

        # Letters, numbers, combining marks and underscore
        is_word_char = (
            category.startswith("L")
            or category.startswith("N")
            or category.startswith("M")
            or char == "_"
        )

        if is_word_char:
            if current_start is None:
                current_start = i

        else:
            # Finish the previous word
            if current_start is not None:
                tokens.append({
                    "token": text[current_start:i],
                    "start": current_start,
                    "end": i
                })
                current_start = None

            # Preserve punctuation as separate tokens
            if not char.isspace():
                tokens.append({
                    "token": char,
                    "start": i,
                    "end": i + 1
                })

    # Add final word if present
    if current_start is not None:
        tokens.append({
            "token": text[current_start:],
            "start": current_start,
            "end": len(text)
        })

    return tokens


def convert_to_bio(example):

    text = example["text"]
    entities = example["entities"]

    # Unicode-aware tokenization with character offsets
    tokens = tokenize_with_offsets(text)

    bio_tags = []

    for token in tokens:

        assigned_tag = "O"

        for entity in entities:

            # Check whether token overlaps entity span
            if (
                token["start"] < entity["end"]
                and token["end"] > entity["start"]
            ):

                if token["start"] == entity["start"]:
                    assigned_tag = "B-" + entity["label"]
                else:
                    assigned_tag = "I-" + entity["label"]

                break

        bio_tags.append(assigned_tag)

    return {
        "tokens": [t["token"] for t in tokens],
        "ner_tags": bio_tags,
        "language": example["language"],
        "template_id": example["template_id"],
        "text": text
    }


# Regenerate BIO dataset from the updated 900 samples
bio_dataset = [
    convert_to_bio(example)
    for example in dataset
]


# Verify dataset size
assert len(bio_dataset) == 900


# Verify token and BIO tag alignment
for item in bio_dataset:
    assert len(item["tokens"]) == len(item["ner_tags"])


print("BIO conversion completed!")
print("Total examples:", len(bio_dataset))


# Display one example
sample = bio_dataset[0]

print("\nText:")
print(sample["text"])

print("\nTokens and BIO tags:")

for token, tag in zip(sample["tokens"], sample["ner_tags"]):
    print(f"{token:<25} {tag}")


# Explicit Hindi tokenization test
hindi_test = tokenize_with_offsets("मेरा आधार नंबर 0000 0000 0006 है।")

print("\nHindi tokenization test:")
print([x["token"] for x in hindi_test])


# Aadhaar BIO tagging test
aadhaar_example = next(
    x for x in bio_dataset
    if "मेरा आधार नंबर" in x["text"]
)

print("\nHindi Aadhaar example:")
print(aadhaar_example["text"])

for token, tag in zip(
    aadhaar_example["tokens"],
    aadhaar_example["ner_tags"]
):
    print(f"{token:<20} {tag}")


BIO conversion completed!
Total examples: 900

Text:
Kavya Nair ka PAN number LMNOP1234Q hai.

Tokens and BIO tags:
Kavya                     B-PERSON
Nair                      I-PERSON
ka                        O
PAN                       O
number                    O
LMNOP1234Q                B-PAN
hai                       O
.                         O

Hindi tokenization test:
['मेरा', 'आधार', 'नंबर', '0000', '0000', '0006', 'है', '।']

Hindi Aadhaar example:
मेरा आधार नंबर 0000 0000 0006 है।
मेरा                 O
आधार                 O
नंबर                 O
0000                 B-AADHAAR
0000                 I-AADHAAR
0006                 I-AADHAAR
है                   O
।                    O


In [39]:
print("Templates:", len(TEMPLATES))
print("Dataset samples:", len(dataset))

aadhaar_count = sum(
    1 for item in dataset
    for entity in item["entities"]
    if entity["label"] == "AADHAAR"
)

print("Aadhaar entities:", aadhaar_count)

print("Language distribution:")
print(pd.Series(
    [item["language"] for item in dataset]
).value_counts())

Templates: 45
Dataset samples: 900
Aadhaar entities: 360
Language distribution:
hinglish    300
en          300
hi          300
Name: count, dtype: int64


In [40]:

# Step 4.2: Group-based train/validation/test split

from sklearn.model_selection import GroupShuffleSplit

EXPECTED_LABELS = {
    "PERSON", "ADDRESS", "PHONE",
    "AADHAAR", "PAN", "EMAIL", "UPI"
}

EXPECTED_LANGUAGES = {"en", "hi", "hinglish"}

# Confirm BIO dataset was regenerated
assert len(dataset) == 900, "Raw dataset must contain 900 samples."
assert len(bio_dataset) == 900, (
    "BIO dataset is outdated. Rerun the BIO conversion cell first."
)

# Extract entity categories from BIO tags
def get_entity_labels(data):
    labels = set()

    for item in data:
        for tag in item["ner_tags"]:
            if tag != "O":
                labels.add(tag.split("-", 1)[-1])

    return labels


# Extract languages
def get_languages(data):
    return {item["language"] for item in data}


# Find templates containing a particular entity
def get_entity_templates(data, entity_label):
    return {
        item["template_id"]
        for item in data
        if any(
            tag.endswith("-" + entity_label)
            for tag in item["ner_tags"]
        )
    }


groups = [item["template_id"] for item in bio_dataset]

valid_split_found = False

# Search for a group split satisfying all coverage conditions
for seed in range(1000):

    split_1 = GroupShuffleSplit(
        n_splits=1,
        test_size=0.15,
        random_state=seed
    )

    train_val_idx, test_idx = next(
        split_1.split(bio_dataset, groups=groups)
    )

    train_val_data = [bio_dataset[i] for i in train_val_idx]
    train_val_groups = [groups[i] for i in train_val_idx]

    split_2 = GroupShuffleSplit(
        n_splits=1,
        test_size=0.15 / 0.85,
        random_state=seed
    )

    train_idx, val_idx = next(
        split_2.split(
            train_val_data,
            groups=train_val_groups
        )
    )

    train_candidate = [train_val_data[i] for i in train_idx]
    val_candidate = [train_val_data[i] for i in val_idx]
    test_candidate = [bio_dataset[i] for i in test_idx]

    # Check entity and language coverage
    coverage_valid = all(
        get_entity_labels(split) == EXPECTED_LABELS
        and get_languages(split) == EXPECTED_LANGUAGES
        for split in [
            train_candidate,
            val_candidate,
            test_candidate
        ]
    )

    # Check Aadhaar template diversity
    train_aadhaar = get_entity_templates(
        train_candidate, "AADHAAR"
    )
    val_aadhaar = get_entity_templates(
        val_candidate, "AADHAAR"
    )
    test_aadhaar = get_entity_templates(
        test_candidate, "AADHAAR"
    )

    if (
        coverage_valid
        and len(train_aadhaar) >= 10
        and len(val_aadhaar) >= 2
        and len(test_aadhaar) >= 2
    ):
        train_data = train_candidate
        val_data = val_candidate
        test_data = test_candidate
        selected_seed = seed
        valid_split_found = True
        break


if not valid_split_found:
    raise ValueError(
        "No valid group split found. "
        "Check entity distribution across templates."
    )


# Verify template separation
train_groups = {x["template_id"] for x in train_data}
val_groups = {x["template_id"] for x in val_data}
test_groups = {x["template_id"] for x in test_data}

assert train_groups.isdisjoint(val_groups)
assert train_groups.isdisjoint(test_groups)
assert val_groups.isdisjoint(test_groups)


# Display split sizes
print("Selected seed:", selected_seed)

print("\nDataset sizes:")
print("Train:", len(train_data))
print("Validation:", len(val_data))
print("Test:", len(test_data))
print("Total:", len(train_data) + len(val_data) + len(test_data))


# Display entity counts per split
print("\nEntity distribution:")

for name, split in [
    ("Train", train_data),
    ("Validation", val_data),
    ("Test", test_data)
]:
    counts = {}

    for item in split:
        for tag in item["ner_tags"]:
            if tag.startswith("B-"):
                label = tag[2:]
                counts[label] = counts.get(label, 0) + 1

    print(f"\n{name}:")
    print(counts)


# Display language distribution
print("\nLanguage distribution:")

for name, split in [
    ("Train", train_data),
    ("Validation", val_data),
    ("Test", test_data)
]:
    language_counts = {}

    for item in split:
        lang = item["language"]
        language_counts[lang] = language_counts.get(lang, 0) + 1

    print(f"{name}: {language_counts}")


# Display Aadhaar template distribution
print("\nUnique Aadhaar templates:")
print("Train:", len(train_aadhaar))
print("Validation:", len(val_aadhaar))
print("Test:", len(test_aadhaar))

print("\nTemplate leakage check: PASSED")
print("All split checks passed!")


Selected seed: 638

Dataset sizes:
Train: 620
Validation: 140
Test: 140
Total: 900

Entity distribution:

Train:
{'EMAIL': 80, 'AADHAAR': 280, 'PERSON': 160, 'ADDRESS': 80, 'PHONE': 100, 'UPI': 60, 'PAN': 40}

Validation:
{'PERSON': 80, 'PAN': 20, 'AADHAAR': 40, 'ADDRESS': 20, 'UPI': 20, 'EMAIL': 20, 'PHONE': 40}

Test:
{'PERSON': 40, 'UPI': 40, 'AADHAAR': 40, 'PHONE': 40, 'PAN': 20, 'ADDRESS': 20, 'EMAIL': 20}

Language distribution:
Train: {'en': 240, 'hi': 180, 'hinglish': 200}
Validation: {'hinglish': 60, 'en': 20, 'hi': 60}
Test: {'hinglish': 40, 'hi': 60, 'en': 40}

Unique Aadhaar templates:
Train: 14
Validation: 2
Test: 2

Template leakage check: PASSED
All split checks passed!


In [41]:

# Step 4.3: Entity distribution analysis

def count_entities(records):
    counts = Counter()

    for record in records:
        for tag in record["ner_tags"]:
            if tag.startswith("B-"):
                counts[tag[2:]] += 1

    return counts

distribution = pd.DataFrame({
    "Train": count_entities(train_data),
    "Validation": count_entities(val_data),
    "Test": count_entities(test_data)
}).fillna(0).astype(int)

display(distribution)

print("\nLanguage distribution:")

for name, records in [
    ("Train", train_data),
    ("Validation", val_data),
    ("Test", test_data)
]:
    print(name, dict(Counter(x["language"] for x in records)))


,Train,Validation,Test
EMAIL,80,20,20
AADHAAR,280,40,40
PERSON,160,80,40
ADDRESS,80,20,20
PHONE,100,40,40
UPI,60,20,40
PAN,40,20,20



Language distribution:
Train {'en': 240, 'hi': 180, 'hinglish': 200}
Validation {'hinglish': 60, 'en': 20, 'hi': 60}
Test {'hinglish': 40, 'hi': 60, 'en': 40}


In [42]:

# Step 4.4: Save processed datasets

def save_json(data, filename):
    output_path = PROCESSED_DIR / filename

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(
            data,
            f,
            ensure_ascii=False,
            indent=2
        )

    print(f"Saved: {output_path}")


save_json(train_data, "train.json")
save_json(val_data, "validation.json")
save_json(test_data, "test.json")

# Save label mappings for model training
label2id = {label: idx for idx, label in enumerate(BIO_LABELS)}
id2label = {idx: label for label, idx in label2id.items()}

with open(PROCESSED_DIR / "label_mapping.json", "w", encoding="utf-8") as f:
    json.dump({
        "label2id": label2id,
        "id2label": id2label
    }, f, ensure_ascii=False, indent=2)

print("\nAll processed datasets saved successfully!")


Saved: /Users/navyaa09/IndiaPII/data/processed/train.json
Saved: /Users/navyaa09/IndiaPII/data/processed/validation.json
Saved: /Users/navyaa09/IndiaPII/data/processed/test.json

All processed datasets saved successfully!


In [43]:
# Verify saved datasets

for filename, expected_size in [
    ("train.json", 620),
    ("validation.json", 140),
    ("test.json", 140)
]:
    with open(PROCESSED_DIR / filename, "r", encoding="utf-8") as f:
        saved_data = json.load(f)

    assert len(saved_data) == expected_size
    print(f"{filename}: {len(saved_data)} samples - VERIFIED")

print("All processed datasets verified successfully!")

train.json: 620 samples - VERIFIED
validation.json: 140 samples - VERIFIED
test.json: 140 samples - VERIFIED
All processed datasets verified successfully!
